# Seamount-edge encounter examples

Find strong local topographic gradients with weak vector coherence, then map and track examples. This distinguishes genuine exposure from cancellation when the eddy overlaps both sides of a feature.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import esp_pv_tools as ept
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}


In [ ]:
data = ept.load_cache()
gaussian = data[data.pv_surface_method.eq("esp_gaussian")].copy()
paths=tilt.Paths(); grid=tilt.load_grid(paths.grid,paths.z_r)
focus=gaussian[gaussian.method.eq("esp_gaussian_2")].copy()
focus["encounter_score"]=focus.PV_grad_topo_p90_local_mag*(1-focus.PV_grad_topo_coherence.clip(0,1))
eddy_rank=(focus.groupby(["Cyc","Eddy"]).encounter_score.max().reset_index()
           .sort_values("encounter_score",ascending=False).groupby("Cyc",group_keys=False).head(2))
chosen=focus.merge(eddy_rank[["Cyc","Eddy"]],on=["Cyc","Eddy"],how="inner")

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
for ax, ((cyc,eddy),track) in zip(axes.flat,chosen.groupby(["Cyc","Eddy"])):
    track=track.sort_values("Day")
    ax.plot(track.Day,track.PV_grad_topo_p90_local_mag,label="p90 local magnitude",color="tab:orange")
    ax2=ax.twinx(); ax2.plot(track.Day,track.PV_grad_topo_coherence,label="coherence",color="tab:blue")
    peak=track.loc[track.encounter_score.idxmax()]; ax.axvline(peak.Day,color="k",ls="--")
    ax.set(title=f"{cyc}{int(eddy)}",xlabel="Day",ylabel="Local magnitude"); ax2.set(ylim=(0,1),ylabel="Coherence")
plt.show()

In [ ]:
for (cyc,eddy), track in chosen.groupby(["Cyc","Eddy"]):
    row=track.loc[track.encounter_score.idxmax()]
    pad=max(60,2.2*row.Rc); inside=((grid.X_grid>=row.xc-pad)&(grid.X_grid<=row.xc+pad)&(grid.Y_grid>=row.yc-pad)&(grid.Y_grid<=row.yc+pad))
    bathy=np.where(grid.mask_rho & inside,grid.h/1000,np.nan)
    fig,ax=plt.subplots(figsize=(6,5.5),constrained_layout=True); cf=ax.contourf(grid.X_grid,grid.Y_grid,bathy,25,cmap="terrain_r")
    tilt.plot_ellipse(ax,row,grid,frac=1,color="cyan",lw=2); tilt.plot_ellipse(ax,row,grid,frac=2,color="white",lw=1)
    ax.scatter(row.xc,row.yc,c="magenta",s=35); ax.set(xlim=(row.xc-pad,row.xc+pad),ylim=(row.yc-pad,row.yc+pad),aspect="equal",
        title=f"{cyc}{int(eddy)}, day {int(row.Day)} | coherence={row.PV_grad_topo_coherence:.2f}",xlabel="x (km)",ylabel="y (km)")
    fig.colorbar(cf,ax=ax,label="Depth (km)"); plt.show()